In [2]:
import re
from ast import literal_eval
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# -----------------------------
# Time array
# -----------------------------
Num_step_list = np.array([3])
initial_t = 0.0
final_t = 1.0
times = []

for num_step in Num_step_list:
    total_t = 0
    delta_t = (final_t - initial_t) / num_step
    times.append(0.0)
    for _ in range(num_step):
        total_t += delta_t
        times.append(total_t)
times = np.array(times)
print("Times:", times)


# 1-qubit Pauli basis (16 operators)
# -----------------------------
Psingle = ["I", "X", "Y", "Z"]
Pauli_combination = [(p1) for p1 in Psingle ]

# Build matrices for each Pauli string
matrices = [SparsePauliOp.from_list([(p, 1)]).to_matrix() for (p) in Psingle]

# -----------------------------
# Function to compute expectation from bitstring
# -----------------------------
##################################################################################
# total 4000 shots
ODR_IZI_run1 = [1.02564103,1.23362671,1.52736214,1.89088048]
#######################################################################################
ODR_factor_1 = np.array([
[1.0, 1.0, 1.0, 1.0],
ODR_IZI_run1, 
ODR_IZI_run1, 
ODR_IZI_run1
])
############################################################################
# total 4000 shots
ODR_IZI_run2 = [1.0178117 ,1.05200156,1.1326897 ,1.38105892]
############################################################################
ODR_factor_2 = np.array([
[1.0, 1.0, 1.0, 1.0],
ODR_IZI_run2, 
ODR_IZI_run2, 
ODR_IZI_run2 
])
##############################################################

exp_values_per_time = np.load("exp_val_run1_run2_N101_A1.npy")
# -----------------------------
# Compute density matrix and entanglement entropy
# -----------------------------
entropies_SA2_set1 = []
total_array_1 =[]
for i, (exp_dict_1,exp_dict_2) in enumerate(exp_values_per_time):
    exp_arr_1 = np.array(exp_dict_1)
    exp_arr_2 = np.array(exp_dict_2)
    # Apply ODR correction if needed:
    corrected = (exp_arr_1 * ODR_factor_1[:, i] + exp_arr_2 * ODR_factor_2[:, i])/2  # Uncomment if you have ODR_factor
    #corrected = exp_arr_1 

    rho = np.zeros((2, 2), dtype=complex)
    exp_sqr = 0
    for M, exp_val in zip(matrices, corrected):
        rho += (1/2) * exp_val * M
        exp_sqr = exp_sqr + (np.abs(exp_val))**2

    #rho = rho / np.trace(rho)
    # Entanglement entropy
    total_1 = -np.log((exp_sqr )/(2))
    #print(f"trace = {np.trace(rho)}")
    vals = np.real(np.linalg.eigvals(rho))

    # --- Moments R1, R2 and R3 ---
    R1 = np.sum(vals)    #  This must be 1 because this is Tr(rho)
    #print(f"R1= {R1}")
    R2 = np.sum(vals**2)
    R3 = np.sum(vals**3)
    
    S_A2 =  - np.log(R2) #- np.real((R2-1)/(R1**2))
    #S_A2 = np.real(-np.sum(vals* np.log(vals)))
    #print(f"EE 2nd and 1st at time ={times[i]:.6f} : {S_A2:.6f}")
    entropies_SA2_set1.append(S_A2)
    #entropies_set1.append(S_A)
    total_array_1.append(total_1)

print("Times =", times)
print("Entropies 2nd way =", np.array(total_array_1))
print("Entropies_SA2  =", np.array(entropies_SA2_set1))


Times: [0.         0.33333333 0.66666667 1.        ]
Times = [0.         0.33333333 0.66666667 1.        ]
Entropies 2nd way = [3.90620152e-04 4.30409307e-01 6.90746134e-01 6.69176489e-01]
Entropies_SA2  = [3.90620152e-04 4.30409307e-01 6.90746134e-01 6.69176489e-01]


In [4]:
data_run1 = np.load("ODR_factor_4Brun1_data.npz")
ODR_factor_4Brun1 = data_run1["ODR_factor_4Brun1"]

data_run2 = np.load("ODR_factor_4Brun2_data.npz")
ODR_factor_4Brun2 = data_run2["ODR_factor_4Brun2"]

print(np.shape(ODR_factor_4Brun1))
print(ODR_factor_4Brun1.shape[2])


ODR_1_sets = []

for b in range(4):   # loop over 4 batches
    
    ODR_factor_set = ODR_factor_4Brun1[b]   # shape (8,4)

    ODR_III = ODR_factor_set[0]
    ODR_IIZ = ODR_factor_set[1]
    ODR_IZI = ODR_factor_set[2]
    ODR_IZZ = ODR_factor_set[3]
    ODR_ZII = ODR_factor_set[4]
    ODR_ZIZ = ODR_factor_set[5]
    ODR_ZZI = ODR_factor_set[6]
    ODR_ZZZ = ODR_factor_set[7]

    ODR_set = np.array([
        ODR_III, 
        ODR_IZI, 
        ODR_IZI, 
        ODR_IZI
    ])

    ODR_1_sets.append(ODR_set)
    print(ODR_set)

ODR_1_set_1 = ODR_1_sets[0]
ODR_1_set_2 = ODR_1_sets[1]
ODR_1_set_3 = ODR_1_sets[2]
ODR_1_set_4 = ODR_1_sets[3]


(4, 8, 4)
4
[[1.         1.         1.         1.        ]
 [1.03305785 1.2607178  1.61290323 2.08946632]
 [1.03305785 1.2607178  1.61290323 2.08946632]
 [1.03305785 1.2607178  1.61290323 2.08946632]]
[[1.         1.         1.         1.        ]
 [1.02669405 1.26070331 1.5576324  1.88341846]
 [1.02669405 1.26070331 1.5576324  1.88341846]
 [1.02669405 1.26070331 1.5576324  1.88341846]]
[[1.         1.         1.         1.        ]
 [1.02669405 1.24634447 1.52905199 1.84852155]
 [1.02669405 1.24634447 1.52905199 1.84852155]
 [1.02669405 1.24634447 1.52905199 1.84852155]]
[[1.         1.         1.         1.        ]
 [1.01626016 1.1705008  1.42045455 1.77042209]
 [1.01626016 1.1705008  1.42045455 1.77042209]
 [1.01626016 1.1705008  1.42045455 1.77042209]]


In [5]:
print(np.shape(ODR_factor_4Brun2))
print(ODR_factor_4Brun2.shape[2])


ODR_2_sets = []

for b in range(4):   # loop over 4 batches
    
    ODR_factor_set = ODR_factor_4Brun2[b]   # shape (8,4)

    ODR_III = ODR_factor_set[0]
    ODR_IIZ = ODR_factor_set[1]
    ODR_IZI = ODR_factor_set[2]
    ODR_IZZ = ODR_factor_set[3]
    ODR_ZII = ODR_factor_set[4]
    ODR_ZIZ = ODR_factor_set[5]
    ODR_ZZI = ODR_factor_set[6]
    ODR_ZZZ = ODR_factor_set[7]

    ODR_set = np.array([
        ODR_III, 
        ODR_IZI, 
        ODR_IZI, 
        ODR_IZI
    ])

    ODR_2_sets.append(ODR_set)

ODR_2_set_1 = ODR_2_sets[0]
ODR_2_set_2 = ODR_2_sets[1]
ODR_2_set_3 = ODR_2_sets[2]
ODR_2_set_4 = ODR_2_sets[3]

(4, 8, 4)
4


In [6]:
# Dividing Original single Batches to 4 different batches, where expectation values are
# evaluated using 1000 shots
#=================================================================

exp_values_sets_run1 = np.load("exp_val_run1_sets_N101_A1.npy" )
exp_values_sets_run2 = np.load("exp_val_run2_sets_N101_A1.npy")

corr_exp_set_1 =[]
for exp_timearra_1 in exp_values_sets_run1:
    corrected_timearr =[]
    for i, exp_1 in enumerate(exp_timearra_1):
        exp_arr_1 = np.array(exp_1)
        # Apply ODR correction if needed:
        corrected = (exp_arr_1 * ODR_1_set_1[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_1.append(corrected_timearr)
corr_exp_set_1 = np.array(corr_exp_set_1)

corr_exp_set_2 =[]
for exp_timearra_1 in exp_values_sets_run1:
    corrected_timearr =[]
    for i, exp_1 in enumerate(exp_timearra_1):
        exp_arr_1 = np.array(exp_1)
        # Apply ODR correction if needed:
        corrected = (exp_arr_1 * ODR_1_set_2[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_2.append(corrected_timearr)
corr_exp_set_2 = np.array(corr_exp_set_2)

corr_exp_set_3 =[]
for exp_timearra_1 in exp_values_sets_run1:
    corrected_timearr =[]
    for i, exp_1 in enumerate(exp_timearra_1):
        exp_arr_1 = np.array(exp_1)
        # Apply ODR correction if needed:
        corrected = (exp_arr_1 * ODR_1_set_3[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_3.append(corrected_timearr)
corr_exp_set_3 = np.array(corr_exp_set_3)

corr_exp_set_4 =[]
for exp_timearra_1 in exp_values_sets_run1:
    corrected_timearr =[]
    for i, exp_1 in enumerate(exp_timearra_1):
        exp_arr_1 = np.array(exp_1)
        # Apply ODR correction if needed:
        corrected = (exp_arr_1 * ODR_1_set_4[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_4.append(corrected_timearr)
corr_exp_set_4 = np.array(corr_exp_set_4)


corr_exp_set_5 =[]
for exp_timearra_2 in exp_values_sets_run2:
    corrected_timearr =[]
    for i, exp_2 in enumerate(exp_timearra_2):
        exp_arr_2 = np.array(exp_2)
        # Apply ODR correction if needed:
        corrected = (exp_arr_2 * ODR_2_set_1[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_5.append(corrected_timearr)
corr_exp_set_5 = np.array(corr_exp_set_5)

corr_exp_set_6 =[]
for exp_timearra_2 in exp_values_sets_run2:
    corrected_timearr =[]
    for i, exp_2 in enumerate(exp_timearra_2):
        exp_arr_2 = np.array(exp_2)
        # Apply ODR correction if needed:
        corrected = (exp_arr_2 * ODR_2_set_2[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_6.append(corrected_timearr)
corr_exp_set_6 = np.array(corr_exp_set_6)

corr_exp_set_7 =[]
for exp_timearra_2 in exp_values_sets_run2:
    corrected_timearr =[]
    for i, exp_2 in enumerate(exp_timearra_2):
        exp_arr_2 = np.array(exp_2)
        # Apply ODR correction if needed:
        corrected = (exp_arr_2 * ODR_2_set_3[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_7.append(corrected_timearr)
corr_exp_set_7 = np.array(corr_exp_set_7)

corr_exp_set_8 =[]
for exp_timearra_2 in exp_values_sets_run2:
    corrected_timearr =[]
    for i, exp_2 in enumerate(exp_timearra_2):
        exp_arr_2 = np.array(exp_2)
        # Apply ODR correction if needed:
        corrected = (exp_arr_2 * ODR_2_set_4[:, i] )
        corrected_timearr.append(corrected)
    corr_exp_set_8.append(corrected_timearr)
corr_exp_set_8 = np.array(corr_exp_set_8)
print(corr_exp_set_8.shape)


exp_val_sets = np.vstack([corr_exp_set_1, corr_exp_set_2, corr_exp_set_3, corr_exp_set_4,
                          corr_exp_set_5, corr_exp_set_6, corr_exp_set_7, corr_exp_set_8])

print(exp_val_sets.shape)
print(exp_val_sets.shape[0])

mean_exp = np.mean(exp_val_sets, axis=0)        # (4, 64)
std_exp  = np.std(exp_val_sets, axis=0, ddof=1)/ np.sqrt(8)
print(mean_exp.shape)
print(std_exp.shape)

(4, 4, 4)
(32, 4, 4)
32
(4, 4)
(4, 4)


In [7]:
print("mean_exp shape:", mean_exp.shape)
print("std_exp shape :", std_exp.shape)
entropyS2 =[]
errorS2 = []
for i, (mean_exp_timearry, std_exp_timearry) in enumerate(zip(mean_exp, std_exp)):
    mean_exp_t = np.array(mean_exp_timearry)
    std_exp_t  = np.array(std_exp_timearry)

    exp_sqr = 0
    delta_sqr = 0
    for exp_val, del_exp in zip( mean_exp_t, std_exp_t):
        # To get Tr(rho^2) sum the exp_val^2
        exp_sqr += (np.abs(exp_val))**2
        # To get error in Tr(rho^2) sum the (exp_val* delta_exp_val)**2
        delta_sqr += (exp_val**2)*(del_exp**2)

    Tr_rho2 = (1/2)*exp_sqr
    delta_Tr_rho2 = (2/2)*np.sqrt(delta_sqr)
    # Renyi entropy defination: -log(Tr[rho^2])
    total_S2 = - np.log(Tr_rho2)

    #Error in Renyi entropy defination: - (delta_Tr_rho2)/(Tr_rho2)
    total_err = (delta_Tr_rho2)/(Tr_rho2)

    # Etropy and Error for t = 0,1/3, 2/3, 1
    entropyS2.append(total_S2)
    errorS2.append(total_err)

entropyS2 = np.array(entropyS2)
errorS2  =  np.array(errorS2)

print(f"Renyi Entropy = {np.array2string(entropyS2, separator =',' )}")
print(f"ErrorRenyi Ent= {np.array2string(errorS2, separator =',' )}")

mean_exp shape: (4, 4)
std_exp shape : (4, 4)
Renyi Entropy = [3.69890102e-04,4.30218598e-01,6.90742878e-01,6.69082433e-01]
ErrorRenyi Ent= [0.00310587,0.00979707,0.00144319,0.00660968]


In [ ]:
Entropies_SA2 = [3.90620152e-04 4.30409307e-01 6.90746134e-01 6.69176489e-01]